# Lab | Langchain Evaluation

## Intro

Pick different sets of data and re-run this notebook. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications.

What did you learn? - Let's discuss that in class

## LangChain: Evaluation

### Outline:

* Example generation
* Manual evaluation (and debuging)
* LLM-assisted evaluation

In [1]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY') 

### Example 1

#### Create our QandA application

In [2]:
from langchain.chains import RetrievalQA
from langchain_community.llms import OpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.document_loaders import CSVLoader, TextLoader
from langchain.indexes import VectorstoreIndexCreator
from langchain.vectorstores import DocArrayInMemorySearch
from langchain.chains import LLMChain
from langchain.chat_models import ChatOpenAI






In [5]:
file = r"C:\Users\nicho\Documents\ironhack\Week 19\lab-langchain-evaluation\data\OutdoorClothingCatalog_1000.csv"
loader = CSVLoader(file_path=file)
data = loader.load()

In [6]:
!pip install --upgrade --force-reinstall sentence-transformers

  Using cached sentence_transformers-5.0.0-py3-none-any.whl.metadata (16 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached torch-2.7.1-cp310-cp310-win_amd64.whl.metadata (28 kB)
  Using cached scikit_learn-1.7.1-cp310-cp310-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.15.3-cp310-cp310-win_amd64.whl.metadata (60 kB)
  Using cached pillow-11.3.0-cp310-cp310-win_amd64.whl.metadata (9.2 kB)
  Using cached typing_extensions-4.14.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached filelock-3.18.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl.metadata (60 kB)
  Using cached packaging-25.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached PyYAML-6.0.2-cp310-cp310-win_amd64.whl.metadata (2.1 kB)
  Using cached regex-2024.11.6-cp310-cp310-win_amd64.whl.metadata (41 kB)
  Using cached requests-2.32.4-py3-none-any.whl.metadata (4.9 kB)
  Using cached tokenizers-0.21.2-cp39-abi3-win_amd64.whl.metadata (6.9 kB)
  Using 

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2025.7.0 which is incompatible.


In [8]:
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs = {'device': 'cpu'})
).from_loaders([loader])

c:\Users\nicho\anaconda3\envs\audioenv\lib\site-packages\pydantic\_migration.py:283: UserWarning: `pydantic.error_wrappers:ValidationError` has been moved to `pydantic:ValidationError`.
  warnings.warn(f'`{import_path}` has been moved to `{new_location}`.')
c:\Users\nicho\anaconda3\envs\audioenv\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [9]:
llm = ChatOpenAI(temperature = 0.0)
qa = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    retriever=index.vectorstore.as_retriever(), 
    verbose=True,
    chain_type_kwargs = {
        "document_separator": "<<<<>>>>>"
    }
)

C:\Users\nicho\AppData\Local\Temp\ipykernel_13328\3556945295.py:1: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(temperature = 0.0)


#### Coming up with test datapoints

In [17]:
from langchain.schema import Document

In [18]:
data[10]


Document(metadata={'source': 'C:\\Users\\nicho\\Documents\\ironhack\\Week 19\\lab-langchain-evaluation\\data\\OutdoorClothingCatalog_1000.csv', 'row': 10}, page_content=": 10\nname: Cozy Comfort Pullover Set, Stripe\ndescription: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.\r\n\r\nSize & Fit\r\n- Pants are Favorite Fit: Sits lower on the waist.\r\n- Relaxed Fit: Our most generous fit sits farthest from the body.\r\n\r\nFabric & Care\r\n- In the softest blend of 63% polyester, 35% rayon and 2% spandex.\r\n\r\nAdditional Features\r\n- Relaxed fit top with raglan sleeves and rounded hem.\r\n- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.\r\n\r\nImported.")

In [19]:
data[11]



Document(metadata={'source': 'C:\\Users\\nicho\\Documents\\ironhack\\Week 19\\lab-langchain-evaluation\\data\\OutdoorClothingCatalog_1000.csv', 'row': 11}, page_content=': 11\nname: Ultra-Lofty 850 Stretch Down Hooded Jacket\ndescription: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of motion. With a slightly fitted style that falls at the hip and best with a midweight layer, this jacket is suitable for light activity up to 20° and moderate activity up to -30°. The soft and durable 100% polyester shell offers complete windproof protection and is insulated with warm, lofty goose down. Other features include welded baffles for a no-stitch construction and excellent stretch, an adjustable hood, an interior media port and mesh stash pocket and a hem drawcord. Machine wash and dry. Imported.')

#### Hard-coded examples

In [20]:
from langchain.prompts import PromptTemplate

In [21]:
from langchain.prompts import PromptTemplate
from langchain.schema import BaseOutputParser
from pydantic import BaseModel, Field

examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set\
        have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty \
        850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

# Define the prompt template
prompt_template = PromptTemplate(
    input_variables=["query"],
    template="Examples:\n"
             "1. Query: Do the Cozy Comfort Pullover Set have side pockets?\n"
             "   Answer: Yes\n"
             "2. Query: What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?\n"
             "   Answer: The DownTek collection\n"
             "Query: {query}\n"
             "Answer:"
)

# Define the output model
class Answer(BaseModel):
    answer: str = Field(description="The answer to the query")

# Create the output parser
class AnswerOutputParser(BaseOutputParser):
    def parse(self, text: str) -> Answer:
        # Split the response to get the answer
        answer = text.strip().split("Answer:")[-1].strip()
        return Answer(answer=answer)

# Initialize the LLM
# llm = OpenAI()
llm = ChatOpenAI()

# Create the LLMChain
llm_chain = LLMChain(
    llm=llm,
    prompt=prompt_template,
    output_parser=AnswerOutputParser()
)

# Example query
query = "Is the Cozy Comfort Pullover Set available in different colors?"

# Run the chain
result = llm_chain.run({"query": query})

# Print the result
print(result)


C:\Users\nicho\AppData\Local\Temp\ipykernel_13328\1856079041.py:46: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(
C:\Users\nicho\AppData\Local\Temp\ipykernel_13328\1856079041.py:56: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = llm_chain.run({"query": query})


answer='Yes, it is available in multiple colors such as gray, navy, and black.'


#### LLM-Generated examples

In [22]:
from langchain.evaluation.qa import QAGenerateChain

In [23]:
example_gen_chain = QAGenerateChain.from_llm(ChatOpenAI())

In [24]:
llm_chain = LLMChain(llm=llm, prompt=prompt_template)

In [25]:
new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data[:5]]
)

C:\Users\nicho\AppData\Local\Temp\ipykernel_13328\3125874183.py:1: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  new_examples = example_gen_chain.apply_and_parse(


In [26]:
new_examples[0]

{'qa_pairs': {'query': "What is the recommended sizing guidance for the Women's Campside Oxfords based on the information provided in the document?",
  'answer': 'The document states that for half sizes not offered, customers should order up to the next whole size.'}}

In [27]:
data[0]

Document(metadata={'source': 'C:\\Users\\nicho\\Documents\\ironhack\\Week 19\\lab-langchain-evaluation\\data\\OutdoorClothingCatalog_1000.csv', 'row': 0}, page_content=": 0\nname: Women's Campside Oxfords\ndescription: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. \r\n\r\nSize & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. \r\n\r\nSpecs: Approx. weight: 1 lb.1 oz. per pair. \r\n\r\nConstruction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. \r\n\r\nQuestions? Please contact us for any inquiries.")

In [28]:
d_flattened = [data['qa_pairs'] for data in new_examples]
d_flattened

[{'query': "What is the recommended sizing guidance for the Women's Campside Oxfords based on the information provided in the document?",
  'answer': 'The document states that for half sizes not offered, customers should order up to the next whole size.'},
 {'query': 'What are the dimensions of the small and medium sizes of the Recycled Waterhog Dog Mat, Chevron Weave?',
  'answer': 'The small size has dimensions of 18" x 28", while the medium size has dimensions of 22.5" x 34.5".'},
 {'query': "What are some key features of the Infant and Toddler Girls' Coastal Chill Swimsuit, Two-Piece as described in the document?",
  'answer': 'Some key features of the swimsuit include bright colors, ruffles, exclusive whimsical prints, four-way-stretch and chlorine-resistant fabric, UPF 50+ rated fabric for high sun protection, crossover no-slip straps, fully lined bottom for secure fit and maximum coverage, and the recommendation of machine wash and line dry for best results.'},
 {'query': 'What 

#### Combine examples

In [29]:
# examples += new_example
examples += d_flattened

In [30]:
examples[0]

{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
 'answer': 'Yes'}

In [33]:
qa.invoke(examples[0]["query"])



> Entering new RetrievalQA chain...


c:\Users\nicho\anaconda3\envs\audioenv\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)



> Finished chain.


{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
 'result': 'Based on the provided context, the Cozy Comfort Pullover Set does not have side pockets. It has side seam pockets, a back zip pocket, two elastic mesh water bottle pockets, and a top compartment with a pocket with a double-seal zipper for quick access.'}

### Manual Evaluation - Fun part

In [34]:
import langchain
langchain.debug = True

In [37]:
qa.invoke("Do any products have a hood?")


[chain/start] [chain:RetrievalQA] Entering Chain run with input:
{
  "query": "Do any products have a hood?"
}
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain] Entering Chain run with input:
[inputs]
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain > chain:LLMChain] Entering Chain run with input:
{
  "question": "Do any products have a hood?",
  "context": ": 297\nname: Camo Trail Strider Midweight Quarter-Zip Pullover Hoodie\ndescription: This camo quarter-zip hoodie is an extremely versatile mid layer that's great for all hunting seasons. It's made from warm, breathable fleece and features an innovative lightweight hood with a built-in face mask. \r\n\r\nFabric & Care: 100% polyester. Machine wash and dry. \r\n\r\nAdditional Features: An excellent layering piece for cold weather. Wind-resistant hard-shell fleece is quiet and resists water, debris and snags. Super-soft fleece interior is warm and can be worn over layers or next-to-skin. Unique lightweight hood

c:\Users\nicho\anaconda3\envs\audioenv\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


[llm/end] [chain:RetrievalQA > chain:StuffDocumentsChain > chain:LLMChain > llm:ChatOpenAI] s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "Yes, the following products have a hood:\n1. Camo Trail Strider Midweight Quarter-Zip Pullover Hoodie\n2. Mountain Haven Hoodie\n3. Camo Trekker Midweight Quarter-Zip Pullover Hoodie\n4. Women's Stop Flies Hoodie",
        "generation_info": {
          "finish_reason": "stop",
          "logprobs": null
        },
        "type": "ChatGeneration",
        "message": {
          "lc": 1,
          "type": "constructor",
          "id": [
            "langchain",
            "schema",
            "messages",
            "AIMessage"
          ],
          "kwargs": {
            "content": "Yes, the following products have a hood:\n1. Camo Trail Strider Midweight Quarter-Zip Pullover Hoodie\n2. Mountain Haven Hoodie\n3. Camo Trekker Midweight Quarter-Zip Pullover Hoodie\n4. Women's Stop Flies Hoodie",
            "

{'query': 'Do any products have a hood?',
 'result': "Yes, the following products have a hood:\n1. Camo Trail Strider Midweight Quarter-Zip Pullover Hoodie\n2. Mountain Haven Hoodie\n3. Camo Trekker Midweight Quarter-Zip Pullover Hoodie\n4. Women's Stop Flies Hoodie"}

In [38]:
# Turn off the debug mode
langchain.debug = False

### LLM assisted evaluation

In [39]:
examples += d_flattened

In [40]:
examples

[{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
  'answer': 'Yes'},
 {'query': 'What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?',
  'answer': 'The DownTek collection'},
 {'query': "What is the recommended sizing guidance for the Women's Campside Oxfords based on the information provided in the document?",
  'answer': 'The document states that for half sizes not offered, customers should order up to the next whole size.'},
 {'query': 'What are the dimensions of the small and medium sizes of the Recycled Waterhog Dog Mat, Chevron Weave?',
  'answer': 'The small size has dimensions of 18" x 28", while the medium size has dimensions of 22.5" x 34.5".'},
 {'query': "What are some key features of the Infant and Toddler Girls' Coastal Chill Swimsuit, Two-Piece as described in the document?",
  'answer': 'Some key features of the swimsuit include bright colors, ruffles, exclusive whimsical prints, four-way-stretch and chlorine-resistant 

In [41]:
predictions = qa.batch(examples)



> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


c:\Users\nicho\anaconda3\envs\audioenv\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
c:\Users\nicho\anaconda3\envs\audioenv\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
c:\Users\nicho\anaconda3\envs\audioenv\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
c:\Users\nicho\anaconda3\envs\audioenv\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  retur


> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.


In [42]:
predictions

[{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
  'answer': 'Yes',
  'result': 'Based on the provided context, the Cozy Comfort Pullover Set does not have side pockets. It has side seam pockets, a back zip pocket, two elastic mesh water bottle pockets, and a top compartment with a pocket with a double-seal zipper for quick access.'},
 {'query': 'What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?',
  'answer': 'The DownTek collection',
  'result': 'The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.'},
 {'query': "What is the recommended sizing guidance for the Women's Campside Oxfords based on the information provided in the document?",
  'answer': 'The document states that for half sizes not offered, customers should order up to the next whole size.',
  'result': "Based on the information provided in the document, the sizing guidance for the Women's Campside Oxfords is as follows:\n\n- Women's Small/Mediu

In [43]:
from langchain.evaluation.qa import QAEvalChain

In [44]:
llm = ChatOpenAI(temperature=0)
eval_chain = QAEvalChain.from_llm(llm)

In [45]:
graded_outputs = eval_chain.evaluate(examples, predictions)

In [46]:
graded_outputs

[{'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'}]

In [47]:
for i, eg in enumerate(examples):
    print(f"Example {i}:")
    print("Question: " + predictions[i]['query'])
    print("Real Answer: " + predictions[i]['answer'])
    print("Predicted Answer: " + predictions[i]['result'])
    # print("Predicted Grade: " + graded_outputs[i]['text'])
    print()

Example 0:
Question: Do the Cozy Comfort Pullover Set        have side pockets?
Real Answer: Yes
Predicted Answer: Based on the provided context, the Cozy Comfort Pullover Set does not have side pockets. It has side seam pockets, a back zip pocket, two elastic mesh water bottle pockets, and a top compartment with a pocket with a double-seal zipper for quick access.

Example 1:
Question: What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?
Real Answer: The DownTek collection
Predicted Answer: The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.

Example 2:
Question: What is the recommended sizing guidance for the Women's Campside Oxfords based on the information provided in the document?
Real Answer: The document states that for half sizes not offered, customers should order up to the next whole size.
Predicted Answer: Based on the information provided in the document, the sizing guidance for the Women's Campside Oxfords is as follows

### Example 2
One can also easily evaluate your QA chains with the metrics offered in ragas

In [51]:
from langchain_community.embeddings import HuggingFaceEmbeddings
loader = TextLoader(r"C:\Users\nicho\Documents\ironhack\Week 19\lab-langchain-evaluation\data\nyc_text.txt")
index = VectorstoreIndexCreator(embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs = {'device': 'cpu'})).from_loaders([loader])


llm = ChatOpenAI(temperature= 0)
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=index.vectorstore.as_retriever(),
    return_source_documents=True,
)

C:\Users\nicho\AppData\Local\Temp\ipykernel_13328\3154549768.py:3: UserWarning: Using InMemoryVectorStore as the default vectorstore.This memory store won't persist data. You should explicitlyspecify a vectorstore when using VectorstoreIndexCreator
  index = VectorstoreIndexCreator(embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs = {'device': 'cpu'})).from_loaders([loader])
c:\Users\nicho\anaconda3\envs\audioenv\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [52]:
# testing it out

question = "How did New York City get its name?"
result = qa_chain.invoke({"query": question})
result["result"]

c:\Users\nicho\anaconda3\envs\audioenv\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


'New York City was originally named New Amsterdam by Dutch colonists in 1626. When the city came under British control in 1664, it was renamed New York after King Charles II of England granted the lands to his brother, the Duke of York. The city has been continuously named New York since November 1674.'

In [53]:
result

{'query': 'How did New York City get its name?',
 'result': 'New York City was originally named New Amsterdam by Dutch colonists in 1626. When the city came under British control in 1664, it was renamed New York after King Charles II of England granted the lands to his brother, the Duke of York. The city has been continuously named New York since November 1674.',
 'source_documents': [Document(id='25e9402c-16b7-4d0e-9159-46bbe6d59e03', metadata={'source': 'C:\\Users\\nicho\\Documents\\ironhack\\Week 19\\lab-langchain-evaluation\\data\\nyc_text.txt'}, page_content='The city and its metropolitan area constitute the premier gateway for legal immigration to the United States. As many as 800 languages are spoken in New York, making it the most linguistically diverse city in the world. New York City is home to more than 3.2 million residents born outside the U.S., the largest foreign-born population of any city in the world as of 2016.New York City traces its origins to a trading post founde

Now in order to evaluate the qa system we generated a few relevant questions. We've generated a few question for you but feel free to add any you want.

In [56]:
eval_questions = [
    "What is the population of New York City as of 2020?",
    "Which borough of New York City has the highest population?",
    "What is the economic significance of New York City?",
    "How did New York City get its name?",
    "What is the significance of the Statue of Liberty in New York City?",
]

eval_answers = [
    "8,804,190",
    "Brooklyn",
    "New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter.",
    "New York City got its name when it came under British control in 1664. King Charles II of England granted the lands to his brother, the Duke of York, who named the city New York in his own honor.",
    "The Statue of Liberty in New York City holds great significance as a symbol of the United States and its ideals of liberty and peace. It greeted millions of immigrants who arrived in the U.S. by ship in the late 19th and early 20th centuries, representing hope and freedom for those seeking a better life. It has since become an iconic landmark and a global symbol of cultural diversity and freedom.",
]

examples = [
    {"query": q, "ground_truths": [eval_answers[i]]}
    for i, q in enumerate(eval_questions)
]

In [57]:
examples

[{'query': 'What is the population of New York City as of 2020?',
  'ground_truths': ['8,804,190']},
 {'query': 'Which borough of New York City has the highest population?',
  'ground_truths': ['Brooklyn']},
 {'query': 'What is the economic significance of New York City?',
  'ground_truths': ["New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter."]},
 {'query': 'How did New York City

#### Introducing RagasEvaluatorChain

`RagasEvaluatorChain` creates a wrapper around the metrics ragas provides (documented [here](https://github.com/explodinggradients/ragas/blob/main/docs/metrics.md)), making it easier to run these evaluation with langchain and langsmith.

The evaluator chain has the following APIs

- `__call__()`: call the `RagasEvaluatorChain` directly on the result of a QA chain.
- `evaluate()`: evaluate on a list of examples (with the input queries) and predictions (outputs from the QA chain). 
- `evaluate_run()`: method implemented that is called by langsmith evaluators to evaluate langsmith datasets.

lets see each of them in action to learn more.

In [58]:
result = qa_chain.invoke({"query": eval_questions[1]})
result["result"]

c:\Users\nicho\anaconda3\envs\audioenv\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


'Manhattan (New York County) has the highest population density of any borough in New York City.'

In [59]:
key_mapping = {
    "query": "question",
    "result": "answer",
    "source_documents": "contexts"
}

result_updated = {}
for old_key, new_key in key_mapping.items():
    if old_key in result:
        result_updated[new_key] = result[old_key]


In [60]:
result_updated

{'question': 'Which borough of New York City has the highest population?',
 'answer': 'Manhattan (New York County) has the highest population density of any borough in New York City.',
 'contexts': [Document(id='6397295a-d3a4-4c1b-81fb-48128baf76c3', metadata={'source': 'C:\\Users\\nicho\\Documents\\ironhack\\Week 19\\lab-langchain-evaluation\\data\\nyc_text.txt'}, page_content="New York City is the most populous city in the United States, with 8,804,190 residents incorporating more immigration into the city than outmigration since the 2010 United States census. More than twice as many people live in New York City as compared to Los Angeles, the second-most populous U.S. city; and New York has more than three times the population of Chicago, the third-most populous U.S. city. New York City gained more residents between 2010 and 2020 (629,000) than any other U.S. city, and a greater amount than the total sum of the gains over the same decade of the next four largest U.S. cities, Los Ang

In [61]:
!pip install --no-cache-dir recordclass

In [62]:
!pip install ragas==0.1.9

  Using cached fsspec-2025.3.0-py3-none-any.whl.metadata (11 kB)
Using cached fsspec-2025.3.0-py3-none-any.whl (193 kB)

  Attempting uninstall: fsspec

    Found existing installation: fsspec 2025.7.0

    Uninstalling fsspec-2025.7.0:

   -------------------- ------------------- 2/4 [fsspec]
      Successfully uninstalled fsspec-2025.7.0
   -------------------- ------------------- 2/4 [fsspec]
   -------------------- ------------------- 2/4 [fsspec]
   ------------------------------ --------- 3/4 [ragas]
   ---------------------------------------- 4/4 [ragas]



In [63]:
from ragas.integrations.langchain import EvaluatorChain 
# from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_relevancy,
    context_recall,
)

# create evaluation chains
faithfulness_chain   = EvaluatorChain(metric=faithfulness)
answer_rel_chain     = EvaluatorChain(metric=answer_relevancy)
context_rel_chain    = EvaluatorChain(metric=context_relevancy)
context_recall_chain = EvaluatorChain(metric=context_recall)

c:\Users\nicho\anaconda3\envs\audioenv\lib\site-packages\ragas\metrics\__init__.py:1: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  from ragas.metrics._answer_correctness import AnswerCorrectness, answer_correctness
c:\Users\nicho\anaconda3\envs\audioenv\lib\site-packages\ragas\metrics\__init__.py:4: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be us

1. `__call__()`

Directly run the evaluation chain with the results from the QA chain. Do note that metrics like context_relevancy and faithfulness require the `source_documents` to be present.

In [64]:
# Recheck the result that we are going to validate.
result

{'query': 'Which borough of New York City has the highest population?',
 'result': 'Manhattan (New York County) has the highest population density of any borough in New York City.',
 'source_documents': [Document(id='6397295a-d3a4-4c1b-81fb-48128baf76c3', metadata={'source': 'C:\\Users\\nicho\\Documents\\ironhack\\Week 19\\lab-langchain-evaluation\\data\\nyc_text.txt'}, page_content="New York City is the most populous city in the United States, with 8,804,190 residents incorporating more immigration into the city than outmigration since the 2010 United States census. More than twice as many people live in New York City as compared to Los Angeles, the second-most populous U.S. city; and New York has more than three times the population of Chicago, the third-most populous U.S. city. New York City gained more residents between 2010 and 2020 (629,000) than any other U.S. city, and a greater amount than the total sum of the gains over the same decade of the next four largest U.S. cities, Lo

**Faithfulness**

In [68]:
# ✅ Convert context Documents to plain strings (if needed)
if isinstance(result_updated.get("contexts", [])[0], dict) or hasattr(result_updated.get("contexts", [])[0], "page_content"):
    result_updated["contexts"] = [doc.page_content if hasattr(doc, "page_content") else str(doc) for doc in result_updated["contexts"]]

# ✅ Run the async faithfulness evaluation
eval_result = await faithfulness_chain.ainvoke(result_updated)

# ✅ Inspect full output and safely extract score
print("🔍 Eval Result Keys:", eval_result.keys())
faithfulness_score = eval_result.get("faithfulness") or eval_result.get("faithfulness_score", "Not returned")
print("✅ Faithfulness Score:", faithfulness_score)



🔍 Eval Result Keys: dict_keys(['question', 'answer', 'contexts', 'faithfulness'])
✅ Faithfulness Score: 0.5


High faithfulness_score means that there are exact consistency between the source documents and the answer.

You can check lower faithfulness scores by changing the result (answer from LLM) or source_documents to something else.

In [85]:
# 🧪 Define a custom fake answer to test faithfulness
fake_answer = "The population of NYC is 3 million and it has no public transport system."

# 🧾 Create a complete, valid evaluation input from the NYC dataset
fake_input = {
    "question": "What can you tell me about New York City's population and infrastructure?",
    "answer": fake_answer,
    "contexts": [doc.page_content if hasattr(doc, "page_content") else str(doc) for doc in result.get("source_documents", [])]
}

# ✅ Run async faithfulness evaluation
eval_result = await faithfulness_chain.ainvoke(fake_input)

# 📊 Display results
print("🔍 Question:", fake_input["question"])
print("🧪 Fake Answer:", fake_input["answer"])
print("📈 Faithfulness Score:", eval_result.get("faithfulness", "N/A"))


🔍 Question: What can you tell me about New York City's population and infrastructure?
🧪 Fake Answer: The population of NYC is 3 million and it has no public transport system.
📈 Faithfulness Score: 0.0


In [86]:
# 🧪 NYC-related fake answers for testing
fake_answers = [
    "New York City has a population of 3 million.",  # Likely false
    "The city is the most densely populated in the U.S.",  # Likely true
    "There are more cows than people in NYC.",  # Definitely false
]

# 🔁 Evaluate each fake answer for faithfulness
for answer in fake_answers:
    fake_input = {
        "question": "What can you tell me about New York City?",
        "answer": answer,
        "contexts": [doc.page_content if hasattr(doc, "page_content") else str(doc) for doc in result.get("source_documents", [])]
    }
    eval_result = await faithfulness_chain.ainvoke(fake_input)
    score = eval_result.get("faithfulness", "N/A")
    print(f"🧪 Answer: {answer}\n📈 Faithfulness Score: {score}\n{'-'*50}")


🧪 Answer: New York City has a population of 3 million.
📈 Faithfulness Score: 0.0
--------------------------------------------------
🧪 Answer: The city is the most densely populated in the U.S.
📈 Faithfulness Score: 0.0
--------------------------------------------------
🧪 Answer: There are more cows than people in NYC.
📈 Faithfulness Score: 0.0
--------------------------------------------------


**Context Relevancy**

In [ ]:
ground_truth_answer = "How old is the statue of liberty?"

recall_input = {
    "question": result.get("query", "What features does the jacket have?"),
    "answer": result.get("result", "The jacket is waterproof."),
    "contexts": [doc.page_content for doc in result.get("source_documents", [])],
    "ground_truth": "The jacket is waterproof and wind-resistant, ideal for outdoor use."
}

eval_result = await context_recall_chain.ainvoke(recall_input)
print("📈 Context Recall Score:", eval_result.get("context_recall", "N/A"))



📈 Context Recall Score: 0.0


In [87]:
for i, ctx in enumerate(recall_input["contexts"]):
    print(f"Context {i+1}:\n{ctx}\n{'-'*40}")


Context 1:
New York City is the most populous city in the United States, with 8,804,190 residents incorporating more immigration into the city than outmigration since the 2010 United States census. More than twice as many people live in New York City as compared to Los Angeles, the second-most populous U.S. city; and New York has more than three times the population of Chicago, the third-most populous U.S. city. New York City gained more residents between 2010 and 2020 (629,000) than any other U.S. city, and a greater amount than the total sum of the gains over the same decade of the next four largest U.S. cities, Los Angeles, Chicago, Houston, and Phoenix, Arizona combined. New York City's population is about 44% of New York State's population, and about 39% of the population of the New York metropolitan area. The majority of New York City residents in 2020 (5,141,538, or 58.4%) were living on Long Island, in Brooklyn, or in Queens. The New York City metropolitan statistical area, has

In [88]:
file = r"C:\Users\nicho\Documents\ironhack\Week 19\lab-langchain-evaluation\data\nyc_text.txt"
loader = CSVLoader(file_path=file)
data = loader.load()

High context_recall_score means that the ground truth is present in the source documents.

You can check lower context recall scores by changing the source_documents to something else.

In [91]:
from langchain.schema import Document

# Define fake ground truth and answer
ground_truth_answer = "New York City is the most populous city in the United States."
fake_answer = "NYC is the capital of the United States."  # Intentionally false

# Prepare the input dict for context recall scoring
recall_input = {
    "question": "What can you tell me about New York City?",
    "answer": fake_answer,
    "ground_truth": ground_truth_answer,
    "contexts": ["I love New York"]
}

# Evaluate context recall
eval_result = await context_recall_chain.ainvoke(recall_input)

# Print result
print("📈 Context Recall Score:", eval_result.get("context_recall", "N/A"))


📈 Context Recall Score: 0.0


In [92]:
file = r"C:\Users\nicho\Documents\ironhack\Week 19\lab-langchain-evaluation\data\OutdoorClothingCatalog_1000.csv"
loader = CSVLoader(file_path=file)
data = loader.load()

2. `evaluate()`

Evaluate a list of inputs/queries and the outputs/predictions from the QA chain.

In [ ]:
examples = [
    {"query": "How many people live in New York City?"},
    {"query": "What is Manhattan known for?"},
    {"query": "What borough has the highest population density?"}
]

all_results = []

for ex in examples:

    prediction = qa_chain.invoke(ex["query"])  # or await if async

    eval_input = {
        "question": ex["query"],
        "answer": prediction.get("result", ""),
        "contexts": [doc.page_content for doc in prediction.get("source_documents", [])]
    }

    eval_result = await faithfulness_chain.ainvoke(eval_input)


    score = eval_result.get("faithfulness", "N/A")
    all_results.append({
        "question": ex["query"],
        "answer": eval_input["answer"],
        "score": score
    })

for res in all_results:
    print(f"🔍 Question: {res['question']}")
    print(f"📤 Answer: {res['answer']}")
    print(f"📈 Faithfulness Score: {res['score']}")
    print("-" * 50)


c:\Users\nicho\anaconda3\envs\audioenv\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
c:\Users\nicho\anaconda3\envs\audioenv\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
c:\Users\nicho\anaconda3\envs\audioenv\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


🔍 Question: How many people live in New York City?
📤 Answer: As of 2020, New York City had a population of 8,804,190 residents.
📈 Faithfulness Score: 1.0
--------------------------------------------------
🔍 Question: What is Manhattan known for?
📤 Answer: Manhattan is known for being the cultural, administrative, and financial center of New York City. It is home to Central Park, most of the city's skyscrapers, major multinational corporations, the United Nations headquarters, Wall Street, and important universities. Manhattan is often described as the financial and cultural center of the world.
📈 Faithfulness Score: 1.0
--------------------------------------------------
🔍 Question: What borough has the highest population density?
📤 Answer: Manhattan (New York County) has the highest population density among all the boroughs in New York City.
📈 Faithfulness Score: 0.5
--------------------------------------------------


In [ ]:
predictions = qa_chain.batch(examples)

faithfulness_scores = []
context_recall_scores = []

for ex, pred in zip(examples, predictions):
    eval_input = {
        "question": ex["query"],
        "answer": pred["result"],
        "contexts": [doc.page_content for doc in pred.get("source_documents", [])],
        "ground_truth": ex.get("ground_truth", "")  # optional, only needed for context recall
    }

    # Faithfulness
    faithfulness_result = await faithfulness_chain.ainvoke(eval_input)
    faithfulness_score = faithfulness_result.get("faithfulness", "N/A")
    faithfulness_scores.append(faithfulness_score)

    # Context Recall
    context_recall_result = await context_recall_chain.ainvoke(eval_input)
    context_score = context_recall_result.get("context_recall", "N/A")
    context_recall_scores.append(context_score)

    print(f"🔍 Question: {eval_input['question']}")
    print(f"🧪 Answer: {eval_input['answer']}")
    print(f"📈 Faithfulness Score: {faithfulness_score}")
    print(f"📊 Context Recall Score: {context_score}")
    print("-" * 60)

import pandas as pd
r = pd.DataFrame({
    "question": [ex["query"] for ex in examples],
    "faithfulness": faithfulness_scores,
    "context_recall": context_recall_scores
})

r


c:\Users\nicho\anaconda3\envs\audioenv\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


🔍 Question: How many people live in New York City?
🧪 Answer: As of 2020, New York City had a population of 8,804,190 residents.
📈 Faithfulness Score: 1.0
📊 Context Recall Score: 1.0
------------------------------------------------------------
🔍 Question: What is Manhattan known for?
🧪 Answer: Manhattan is known for being the cultural, administrative, and financial center of New York City. It is home to Central Park, most of the city's skyscrapers, major multinational corporations, the United Nations headquarters, Wall Street, and important universities. Manhattan is often described as the financial and cultural center of the world.
📈 Faithfulness Score: 0.8181818181818182
📊 Context Recall Score: 1.0
------------------------------------------------------------
🔍 Question: What borough has the highest population density?
🧪 Answer: Manhattan (New York County) has the highest population density among the boroughs of New York City.
📈 Faithfulness Score: 1.0
📊 Context Recall Score: 1.0
-----

,question,faithfulness,context_recall
0,How many people live in New York City?,1.000000,1.0
1,What is Manhattan known for?,0.818182,1.0
2,What borough has the highest population density?,1.000000,1.0
